# Processing overview — the signal's journey

This notebook tells the **signal's story** across the processing pipeline, for
one session, end to end:

```
raw fluorescence (YrA = C)  ──►  low-pass denoise (C_lp)  ──►  deconvolved neural activity (calab)
        Minian                        Minian                    CaTune / CaDecon
```

It's the *signal* counterpart to the capstone (which tells the *behavioral*
story — place cells). Here we just look at what each stage did to the data:
footprints + field of view, a few example cells through all three stages, and
the population activity.

> Reads a session's `minian_out/` (from `pipeline_no_deconv.ipynb`) and
> `deconv_out/activity.npy` (from `run_cadecon.py` / `run_catune.py`). Run those
> first, or `python scripts/get_data.py --session <name>`.

## Load a session

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

SESSION = "prerecorded"                       # or "live"
SESS = Path(f"../../data/sessions/{SESSION}")
MINIAN_OUT = SESS / "minian_out"
DECONV_OUT = SESS / "deconv_out"
NEURAL_FPS = 20.0                             # neural rate (Hz); match arena_config.yaml

# --- Minian output: C = raw YrA, C_lp = low-pass denoised, A, max_proj ---
from minian.utilities import open_minian
mn = open_minian(str(MINIAN_OUT))
C    = mn["C"]                                       # raw extracted fluorescence (YrA)
C_lp = mn["C_lp"] if "C_lp" in mn else None          # low-pass denoised (optional)
A        = mn["A"] if "A" in mn else None
max_proj = mn["max_proj"] if "max_proj" in mn else None

# --- Deconvolution output (calab) ---
act_path = DECONV_OUT / "activity.npy"
activity = np.load(act_path) if act_path.exists() else None
if activity is not None and activity.ndim == 1:
    activity = activity.reshape(1, -1)

n_units = int(C.sizes["unit_id"]); n_frames = int(C.sizes["frame"])
unit_ids = [int(u) for u in C.coords["unit_id"].values]
t = np.arange(n_frames) / NEURAL_FPS

print(f"Minian: {n_units} units x {n_frames} frames  (C = raw YrA)")
print(f"  C_lp (low-pass):       {'present' if C_lp is not None else 'MISSING'}")
print(f"  deconvolved activity:  "
      f"{activity.shape if activity is not None else 'MISSING -> run run_cadecon.py / run_catune.py'}")
if activity is not None and activity.shape[0] != n_units:
    print(f"  WARNING: activity has {activity.shape[0]} rows but Minian has {n_units} units "
          f"(rows must align 1:1 with C's unit order).")

## 1 · Footprints & field of view

Spatial footprints (`A`) over the max-projection image — the cells Minian found.

In [ ]:
if A is not None and max_proj is not None:
    A_sum = A.sum("unit_id").compute()
    img = A_sum.data
    img = np.asarray(img.todense() if hasattr(img, "todense") else img)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(np.asarray(max_proj), cmap="gray")
    ax.imshow(np.ma.masked_where(img <= 0, img), cmap="autumn", alpha=0.6)
    ax.set_title(f"{n_units} footprints over max projection"); ax.axis("off")
    plt.show()
else:
    print("A / max_proj not available in this session.")

## 2 · The signal's journey (per cell)

For a few example cells, the same trace through every stage on one time axis:

- **raw `YrA` (= `C`)** — extracted fluorescence, what Minian saved (gray)
- **low-pass `C_lp`** — denoised, no deconvolution (blue)
- **deconvolved neural activity** — from calab (red, right axis)

This is the whole point of the workshop's design: Minian never deconvolves, so
the red trace is produced entirely by calab from the (un-deconvolved) fluorescence.

In [ ]:
n_ex = min(4, n_units)
base = C_lp if C_lp is not None else C
var = base.var("frame").compute().values
order = np.argsort(var)[::-1]
ex = sorted(int(base.coords["unit_id"].values[i]) for i in order[:n_ex])

fig, axes = plt.subplots(n_ex, 1, figsize=(12, 2.2 * n_ex), sharex=True, squeeze=False)
for r, uid in enumerate(ex):
    ax = axes[r, 0]
    ax.plot(t, C.sel(unit_id=uid).values, color="0.75", lw=0.5, label="raw YrA (C)")
    if C_lp is not None:
        ax.plot(t, C_lp.sel(unit_id=uid).values, color="C0", lw=0.9, label="low-pass C_lp")
    ax.set_ylabel(f"unit {uid}\nfluor (a.u.)", fontsize=8)
    ax2 = None
    if activity is not None and uid in unit_ids:
        i = unit_ids.index(uid)
        ax2 = ax.twinx()
        ax2.plot(t, activity[i], color="crimson", lw=0.8, label="deconvolved activity")
        ax2.set_ylabel("activity", color="crimson", fontsize=8)
        ax2.tick_params(axis="y", labelcolor="crimson")
    if r == 0:
        ax.legend(loc="upper left", fontsize=8)
        if ax2 is not None:
            ax2.legend(loc="upper right", fontsize=8)
axes[-1, 0].set_xlabel("time (s)")
fig.suptitle("Signal's journey: raw fluorescence → low-pass denoise → deconvolved activity (calab)")
plt.tight_layout(); plt.show()

## 3 · Population view

All units' deconvolved neural activity as a raster, with the summed population
activity below.

In [ ]:
if activity is not None:
    fig, (a0, a1) = plt.subplots(
        2, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
    )
    im = a0.imshow(
        activity, aspect="auto", cmap="magma", interpolation="nearest",
        extent=[0, n_frames / NEURAL_FPS, n_units, 0],
    )
    a0.set_ylabel("unit"); a0.set_title("Deconvolved neural activity (all units)")
    fig.colorbar(im, ax=a0, label="activity (a.u.)")
    a1.plot(t, activity.sum(0), color="crimson", lw=0.7)
    a1.set_xlabel("time (s)"); a1.set_ylabel("pop. sum")
    plt.tight_layout(); plt.show()
else:
    print("No deconvolved activity (deconv_out/activity.npy) -> population view skipped.")

## Next

This is the signal side of the story. The **capstone**
(`capstone/camap_placecells.ipynb`) takes the same deconvolved activity, fuses
it with the tracked behavior on the neural clock, and asks the behavioral
question: *which of these cells are place cells?*